### P1: Hermite Polynomials

There are two base cases: $H_0(x) = 1$ and $H_1(x) = 2x$, which would simply
become branches of `if` statements in your Python code.

In [ ]:
def hermite(x, n):
    if n == 0:
        return 1
    elif n == 1:
        return 2 * x
    else:
        return 2 * x * hermite(x, n - 1) - 2 * (n - 1) * hermite(x, n - 2)

print(hermite(2, 7))

The given solution is actually very inefficient because the value $H_n(x)$ for some $n$
gets constantly recomputed (exponentially more and more times as $n$ increases).

### P2: Multi-multiplication

Let's walk through the problem. First, we implement the na&iuml;ve method of multiplying $x$ by itself 10000 times.

In [ ]:
x = 1.05
n = 10000
result = 1

for _ in range(n): # Repeat n = 10000 times
    result *= x # Increase the value of result by a factor of x = 1.05

# By the end, result will be multiplied by x 10000 times, thus giving x^10000
print(result)

Simple enough... But imagine if instead of multiplying a real number, we were working with 1024x1024 matrices instead.
Multiplying those 10000 times would be very, *very* slow. Let's think about a better method.

Consider getting the result of $x^8$. Yes we can do 8 multiplications, but we can also just square $x$ 3 times,
thus doing 3 multiplications:
$$
((x^2)^2)^2\ =\ x^8
$$

In fact, any integer power of $x$ can be achieved by only squaring and multiplying by $x$. Recall that:
$$
(x^n)^2\ =\ x^{2n}
$$$$
(x^n) \cdot x\ =\ x^{n+1}
$$
Then, as an example, 10000 can be expressed as:
$$
10000\ =\ ((((1 \cdot 2 \cdot 2 \cdot 2 + 1) \cdot 2 + 1) \cdot 2 + 1) \cdot 2 \cdot 2 \cdot 2 \cdot 2 + 1) \cdot 2 \cdot 2 \cdot 2 \cdot 2
$$
Which is equivalent to its binary expansion:
$$
10000_{10}\ =\ 10011100010000_{2}
$$
Where we start at the leftmost digit. Each time we encounter a 1, add 1 to the result,
then multiply by 2 everytime we step one digit to the right. Thus, $x^{10000}$ can be expressed as:
$$
x^{10000}\ =\ (((((((((((((x^2)^2)^2) \cdot x)^2 \cdot x)^2 \cdot x)^2)^2)^2)^2 \cdot x)^2)^2)^2)^2
$$
With this trick, we only used 17 multiplications to compute $x^{10000}$.

***Wait... So how do we implement this in Python?***

The main difficulty in the implementation is in knowing when to multiply by $x$ before squaring,
which is based on whether the binary digit is a 1 or 0. Also, we need to start at the leftmost binary digit
which is awkward for integers. For ease of implementation, we convert the exponent 10000 into a string in binary
format:

In [ ]:
n = 10000
f'{n:b}'

In this form, it is now more natural to iterate from the left, and also we know if a digit is a 1 or 0.
Then the implementation is fairly straightforward:

In [ ]:
x = 1.05
n = 10000
result = 1

for digit in f'{n:b}': # Loop through all the digits of n in binary from left to right
    result *= result # Square the result first (corresponding to stepping through a binary digit)
    if digit == '1':
        result *= x # Increase the exponent by 1 if current digit is a 1

# Result may be slightly different than above due to floating point imprecision.
print(result)

### P3: Data Analysis

Read the JSON file into arrays:

In [9]:
import json

with open('Experimental_Data.json', 'r') as f:
    data = json.load(f)

data_x = data['X']
data_y = data['Y']

Tabulate the data:

In [ ]:
print('|   X   |   Y   |')
print('|-------|-------|')
for x, y in zip(data_x, data_y):
    print(f'| {x:>5.3f} | {y:>5.1f} |')

The main data analysis part:

In [ ]:
def mean(x_vals):
    return sum(x_vals) / len(x_vals)

def cov(x_vals, y_vals):
    mean_x = mean(x_vals)
    mean_y = mean(y_vals)

    return mean([ (x - mean_x) * (y - mean_y) for x, y in zip(x_vals, y_vals) ])

a = cov(data_x, data_y) / cov(data_x, data_x)
b = mean(data_y) - a * mean(data_x)

print("a =", a)
print("b =", b)
print('Estimation of y(0.25) =', a * 0.25 + b)

### P4: Code Cleanup Exercise

Here's the code I originally wrote for that purpose. It's not the cleanest and actually has some minor differences versus the code snippet
in the main problem.

In [ ]:
import matplotlib.pyplot as plt
from math import sqrt, pi

grav_strength = 11506.1 * 1.
dist_init = 50.
vel_init = 12.

orbital_energy = 0.5*vel_init**2 - grav_strength/dist_init

def calc_grav(x, y):
    dist = sqrt(x*x + y*y)
    norm_x = -x / dist
    norm_y = -y / dist

    g = grav_strength / (dist * dist)
    return g * norm_x, g * norm_y

dt = 0.0001
plot_skip = 15

x = dist_init
y = 0.
vx = 0.
vy = vel_init
ax_prev, ay_prev = calc_grav(x, y)

xhist = [x]
yhist = [y]

while True:
    x += vx*dt + 0.5*ax_prev*dt*dt
    y += vy*dt + 0.5*ay_prev*dt*dt

    xhist.append(x)
    yhist.append(y)

    ax_new, ay_new = calc_grav(x, y)

    vx += 0.5 * (ax_prev + ax_new) * dt
    vy += 0.5 * (ay_prev + ay_new) * dt

    ax_prev = ax_new
    ay_prev = ay_new

    if yhist[-1] < 0 and yhist[-2] >= 0:
        break

steps_taken = len(xhist) - 1 + yhist[-1] / (yhist[-2] - yhist[-1])
period = 2 * steps_taken * dt
opp_dist = (yhist[-1] * xhist[-2] - yhist[-2] * xhist[-1]) / (yhist[-2] - yhist[-1])
semi_major_axis = (opp_dist + dist_init) / 2

if dist_init > opp_dist:
    print('periapsis distance:', opp_dist)
    print('apoapsis distance:', dist_init)
else:
    print('periapsis distance:', dist_init)
    print('apoapsis distance:', opp_dist)
print('semimajor axis:', semi_major_axis)
print('period:', period)

pi_estimate = 0.5 * period * sqrt(grav_strength / semi_major_axis**3)
print('estimate of pi:', pi_estimate)
print('err:', abs(pi - pi_estimate))

plt.gca().set_aspect('equal')
plt.plot(xhist[::plot_skip], yhist[::plot_skip], 'r-')
plt.grid()

plt.show()

In the original `agrav` function:
```python
def agrav(x,y):
    xn=-x/sqrt(x*x+y*y)
    yn=-y/sqrt(x*x+y*y)
    g=GM*m/(x*x+y*y) # <== g = GMm/r^2, gravitational force
    return g*xn,g*yn
```
The returned value is the $x$ and $y$ components of gravitational force, where `g` is the magnitude of the force
(as it was calculated as $GMm/r^2$).

But in the main `while` loop:
```python
while True:
    x+=vx*dt+0.5*ax0*dt*dt
    y+=vy*dt+0.5*ay0*dt*dt
    xs.append(x)
    ys.append(y)
    ax1,ay1=agrav(x,y) # <== But used as acceleration
    vx+=0.5*(ax0+ax1)*dt
    vy+=0.5*(ay0+ay1)*dt
    ax0,ay0=ax1,ay1
```
The output of `agrav` is used directly for the acceleration in the Verlet integration. Thus, either
divide the output by `m`, the mass of the body, or remove the `m` term in the `agrav` function.

### P5: World's Worst Quicksort

In [ ]:
from random import shuffle

def quicksort(lst):
    if len(lst) <= 1: # The base case, a list of 0 or 1 items is already sorted by itself
        return lst

    pivot = lst[0] # Extract the pivot
    rest = lst[1:] # The list without the pivot

    # The left part of the list, less than or equal pivot
    left = [n for n in rest if n <= pivot]
    # The right part of the list, more than pivot
    right = [n for n in rest if n > pivot]

    # Quicksort the individual parts, then join together with pivot in the middle
    return quicksort(left) + [pivot] + quicksort(right)
        

# Create an unsorted list of integers 1 to 20
lst = list(range(1, 21))
shuffle(lst)

print(quicksort(lst))

### P6: Poker Hands

For beginners to Python, I recommend representing a card with a 2-tuple containing its rank and suit.
The exact details are up to you because I will be using a different method instead,
the resulting code will be fairly similar though.

If we represent diamonds as 0, clubs as 1, hearts as 2, and spades as 3.
Furthermore, represent Ace as 1, Jack as 11, Queen as 12, King as 13.
Let $r$ be the rank and $s$ be the suit, then $4(r-1) + s$ specifies
a unique integer for each card from 0 to 51

The functions to get an integer's rank and suit is then:

In [1]:
def rank(c):
    return c // 4 + 1
def suit(c):
    return c % 4

I will also create a function that returns a card's integer representation,
along with defining some constants.

In [5]:
def card(r, s):
    return 4 * (r - 1) + s

ACE = 1
JACK = 11
QUEEN = 12
KING = 13
DIAMOND = 0
CLUB = 1
HEART = 2
SPADE = 3

A simple way to code the counting function here is to create a list of 13 zeroes corresponding to each rank.
Then, iterate through the hand, incrementing the integer in the list corresponding to the rank.
We get the list we want but it contains a bunch of extra `0`s and `1`s. So we filter them before returning.

In [15]:
def count_matching_cards(hand):
    counts = [0] * 13
    for c in hand:
        # Remember that we defined the representation of the rank to start at 1
        # So need to -1 here because list indices start at 0
        counts[rank(c) - 1] += 1
    return [n for n in counts if n > 1]

To detect a flush, we just check if all the suits are equal (i.e. if all the suits are equal the first one).

In [57]:
def is_flush(hand):
    first_suit = suit(hand[0])
    return all(suit(c) == first_suit for c in hand)

In [47]:
# If you know sets you can do this instead
def is_flush(hand):
    return len(set( suit(c) for c in hand )) == 1

To detect straights, we first write a function that detects if a list is in sequence
(i.e. difference of consecutive items is 1)

In [56]:
def in_sequence(lst):
    # Start at index 1 instead of 0
    # because we are comparing consecutive elements
    for i in range(1, len(lst)):
        if lst[i] != lst[i - 1] + 1:
            return False
    return True

In [55]:
# If you are smart with iterators you can do this instead
def in_sequence(lst):
    return all(curr == prev + 1 for prev, curr in zip( lst, lst[1:] ))

When implementing straight detection, the Python builtin `sorted` is very useful.
It returns a sorted version of an iterator. Thus since we defined the ranks to be in straight order above,
a straight is then obtained when the sorted ranks are in sequence.

This is of course with the exception of the Ace, which can also form a straight in the very end.
To account for this, if the ranks do not form a sequence but contains an `ACE = 1` in front,
then we would remove the leading `ACE` and append a `14` to the end of the ranks instead,
then recheck for sequentiality. Note that `14` technically doesn't represent any actual rank,
but would represent the rank of the Ace if it were at the end of a straight.

In [58]:
def is_straight(hand):
    ranks = sorted(rank(c) for c in hand)
    
    if in_sequence(ranks):
        return True
    if ACE not in ranks:
        return False

    ranks.pop(0)
    ranks.append(14)
    return in_sequence(ranks)

Now to actually implement the main categorisation function.
Note that straight flushes are just hands that are both flush and straight.
Royal flushes are straight flushes that have 10, J, Q, K, A.
Most notably, a straight that contains a King and an Ace will always be royal.

In [68]:
def category(hand):
    flush = is_flush(hand)
    straight = is_straight(hand)

    if flush and straight:
        ranks = [rank(c) for c in hand]
        if KING in ranks and ACE in ranks:
            return 0 # Royal flush
        else:
            return 1 # Straight flush
    # Technically we should check four of a kind and full house first
    # But we assume the hand is valid i.e. no repeating cards,
    # In that case, they are all mutually exclusive
    elif flush:
        return 4 # Flush
    elif straight:
        return 5 # Straight

    matching_counts = count_matching_cards(hand)
    if matching_counts == [4]:
        return 2 # Four of a kind
    elif matching_counts == [2, 3] or matching_counts == [3, 2]:
        return 3 # Full house
    elif matching_counts == [3]:
        return 6 # Three of a kind
    elif matching_counts == [2, 2]:
        return 7 # Two pair
    elif matching_counts == [2]:
        return 8 # One pair
    else:
        return 9 # High card


Since the way we represented a card is just as integers 0 to 51, we can thus just use `range` to
generate a list of all cards.

In [82]:
from random import sample

all_cards = list(range(52))

def draw_hand():
    return sample(all_cards, 5)

Then we just perform the many trials to compute all the estimated probabilities.
To keep track of the count, create an array with 10 zeroes. Increment the value
at index `i` where `i` is the result of `category(hand)` which would always correspond
to an array index since we defined the result as integers 0 to 9.

In [ ]:
# PLEASE CHANGE IF YOU DON'T HAVE A VERY GOOD COMPUTER
N = 2_500_000
counts = [0] * 10
# The actual simulation is just this for loop
for _ in range(N):
    hand = draw_hand()
    cat = category(hand)
    counts[cat] += 1

category_names = [
    'Royal flush',
    'Straight flush',
    'Four of a kind',
    'Full house',
    'Flush',
    'Straight',
    'Three of a kind',
    'Two pair',
    'One pair',
    'High card',
]

for catname, count in zip(category_names, counts):
    percentage_prob = 100 * count / N
    print(f'{catname:<15} | {percentage_prob:>8.4f}% | {count}')

### Extra: Code Golf

In [28]:
def h(n,t=1):return n and h(n>>4,0)+chr(n%16+48+(n%16>9)*7)or'0'*t